# Chapter 4: Experimental Setup and Evaluation

This notebook contains all code for Chapter 4 evaluation, including:
- Configuration reporting (Section 4.1)
- Metrics definition (Section 4.2)
- DQN vs Fixed-Time evaluation
- Statistical analysis (Section 4.3)
- Results analysis and comparison

## Setup and Imports

In [2]:
import subprocess
import sys

# Install required packages
packages = ["numpy>=1.24", "torch>=2.0", "pyyaml>=6.0", "scipy>=1.10"]
for package in packages:
    try:
        # Try importing the package to see if it's already installed
        package_name = package.split(">=")[0]
        __import__(package_name)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All dependencies installed")

Installing pyyaml>=6.0...
✅ All dependencies installed


In [3]:
import os
import sys
import yaml
import numpy as np
import torch
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from pathlib import Path
import csv

# SUMO imports
if 'SUMO_HOME' in os.environ:
    tools = os.path.join(os.environ['SUMO_HOME'], 'tools')
    sys.path.append(tools)
else:
    sys.exit("Please declare environment variable 'SUMO_HOME'")

import traci

# Local imports
from src.env.sumo_env import SumoMDPEnv, EnvConfig
from src.dqn.agent import DQNAgent, AgentConfig
from src.baseline.fixed_time_controller import FixedTimeController, FixedTimeConfig

# Statistical analysis
try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("Warning: scipy not available. Statistical tests will be limited.")

print("✅ All imports successful")

✅ All imports successful


## Section 4.1: Configuration Report

Generate experimental configuration details for the thesis.

In [4]:
class ConfigurationReport:
    """Generate Section 4.1 Configuration Report for Chapter 4."""
    
    def __init__(self, config_path: str = "config.yaml"):
        self.config_path = config_path
        with open(config_path, 'r') as f:
            config_data = yaml.safe_load(f)
            self.config = config_data if config_data is not None else {}
    
    def generate_simulation_parameters_table(self) -> str:
        """Generate Table 4.1: SUMO Simulation Parameters."""
        sumo_config = self.config.get('sumo', {})
        
        table = """
Table 4.1: SUMO Simulation Parameters
==========================================
Parameter                    | Value
-----------------------------|------------------
Scenario                     | {} intersection
Step Length                  | {} seconds
Yellow Phase Duration        | {} seconds
Episode Duration             | {} steps
Warmup Period                | {} steps
GUI Mode                     | {}
""".format(
            sumo_config.get('scenario_path', 'data/scenarios/hn_sample/config.sumocfg'),
            sumo_config.get('step_length', 1.0),
            sumo_config.get('yellow_duration', 5),
            sumo_config.get('max_steps', 1000),
            sumo_config.get('warmup_steps', 60),
            "Enabled" if sumo_config.get('gui', False) else "Disabled"
        )
        return table
    
    def generate_vehicle_weighting_table(self) -> str:
        """Generate Table 4.2: Vietnamese Vehicle Weighting."""
        vn = self.config.get('vn_weights', {})
        moto  = vn.get('motorcycle', 0.5)
        car   = vn.get('car', 1.5)
        bus   = vn.get('bus', 2.0)
        truck = vn.get('truck', 2.0)
        table = """
Table 4.2: Vietnamese Vehicle Weighting Factors (PCU-based)
============================================================
Vehicle Type    | Weight Factor | Justification
----------------|---------------|----------------------------------
Motorcycle      | {:.1f}          | Small, 1 car ≈ 3 motorcycles (PCU)
Car             | {:.1f}          | Larger footprint, baseline congestion
Bus             | {:.1f}          | Large vehicle, high capacity
Truck           | {:.1f}          | Large vehicle, freight transport
""".format(moto, car, bus, truck)
        return table
    
    def generate_state_action_table(self) -> str:
        """Generate Table 4.3: State and Action Space Configuration."""
        table = """
Table 4.3: State and Action Space Configuration
================================================
Component           | Description                      | Dimension
--------------------|----------------------------------|------------
State Features      | Queue length per lane            | 8 lanes
                    | Average speed per lane           | 8 lanes
                    | Waiting time per lane            | 8 lanes
                    | Weighted vehicle count per lane  | 8 lanes
Total State Size    |                                  | 32
Action Space        | 4 phases (NS, EW, NSL, EWL)     | 4
Reward Function     | queue-delay penalty              | Scalar
"""
        return table
    
    def generate_baseline_method_table(self) -> str:
        """Generate Table 4.4: Fixed-Time Controller Configuration."""
        table = """
Table 4.4: Fixed-Time Controller Configuration (Asymmetric 160s cycle)
=======================================================================
Parameter              | Value
-----------------------|----------------------------------------------
Control Strategy       | Fixed-Time Cyclic (Asymmetric)
Number of Phases       | 4
EW Green Duration      | 100 seconds  (phase 2 - Main axis)
NS Green Duration      | 50 seconds   (phase 0 - Secondary axis)
Yellow Duration        | 5 seconds    (phases 1 & 3)
Total Cycle Length     | 160 seconds
Phase Sequence         | EW-green(2) -> EW-yellow(3) -> NS-green(0) -> NS-yellow(1)
Adaptation             | None (Static)
"""
        return table
    
    def generate_control_comparison_table(self) -> str:
        """Generate Table 4.5: Control Strategy Comparison."""
        table = """
Table 4.5: Control Strategy Comparison
======================================
Feature              | Fixed-Time        | DQN-Based
---------------------|-------------------|------------------
Adaptability         | None              | Real-time adaptive
Traffic Awareness    | No                | Yes (state-based)
Learning Capability  | No                | Yes (RL-based)
Implementation       | Simple            | Complex
Computational Cost   | Very Low          | Moderate
Optimization         | Pre-determined    | Experience-based
"""
        return table
    
    def save_configuration_report(self, output_path: str = "outputs/chapter4/configuration_report.txt"):
        """Save complete configuration report."""
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'w') as f:
            f.write("="*60 + "\n")
            f.write("CHAPTER 4 - EXPERIMENTAL CONFIGURATION REPORT\n")
            f.write("="*60 + "\n\n")
            
            f.write("SECTION 4.1.1: SIMULATION PARAMETERS\n")
            f.write(self.generate_simulation_parameters_table())
            f.write("\n")
            
            f.write(self.generate_vehicle_weighting_table())
            f.write("\n")
            
            f.write(self.generate_state_action_table())
            f.write("\n")
            
            f.write("SECTION 4.1.2: BASELINE METHOD\n")
            f.write(self.generate_baseline_method_table())
            f.write("\n")
            
            f.write(self.generate_control_comparison_table())
        
        print(f"✅ Configuration report saved to: {output_path}")
    
    def print_configuration_summary(self):
        """Print all configuration tables."""
        print(self.generate_simulation_parameters_table())
        print(self.generate_vehicle_weighting_table())
        print(self.generate_state_action_table())
        print(self.generate_baseline_method_table())
        print(self.generate_control_comparison_table())

# Generate configuration report
config_report = ConfigurationReport("config.yaml")
config_report.print_configuration_summary()
config_report.save_configuration_report()



Table 4.1: SUMO Simulation Parameters
Parameter                    | Value
-----------------------------|------------------
Scenario                     | data/scenarios/hn_sample/config.sumocfg intersection
Step Length                  | 1.0 seconds
Yellow Phase Duration        | 5 seconds
Episode Duration             | 3600 steps
Warmup Period                | 60 steps
GUI Mode                     | Disabled


Table 4.2: Vietnamese Vehicle Weighting Factors (PCU-based)
Vehicle Type    | Weight Factor | Justification
----------------|---------------|----------------------------------
Motorcycle      | 0.5          | Small, 1 car ≈ 3 motorcycles (PCU)
Car             | 1.5          | Larger footprint, baseline congestion
Bus             | 2.0          | Large vehicle, high capacity
Truck           | 2.0          | Large vehicle, freight transport


Table 4.3: State and Action Space Configuration
Component           | Description                      | Dimension
--------------------|--

## Section 4.2: Evaluation Metrics Definition

Define and document the three main evaluation metrics.

In [5]:
class MetricDefinitions:
    """Section 4.2: Evaluation Metrics Definitions."""
    
    @staticmethod
    def average_vehicle_waiting_time() -> str:
        """4.2.1: Average Vehicle Waiting Time (AWT) definition."""
        return """
Section 4.2.1: Average Vehicle Waiting Time (AWT)
==================================================

Definition:
The average time vehicles spend waiting at the intersection during an episode.

Formula:
    AWT = (1/T) × Σ(t=1 to T) W_t

Where:
    T  = Total number of time steps in episode
    W_t = Total waiting time of all vehicles at step t

Unit: seconds

Interpretation:
    - Lower values indicate better performance
    - Measures traffic flow efficiency
    - Directly impacts user experience
"""
    
    @staticmethod
    def queue_length() -> str:
        """4.2.2: Queue Length definition."""
        return """
Section 4.2.2: Queue Length
===========================

Definition:
The number of vehicles stopped or moving very slowly (<0.1 m/s) at the intersection.

Variants:
    1. Average Queue Length (AQL):
       AQL = (1/T) × Σ(t=1 to T) Q_t
    
    2. Maximum Queue Length (MQL):
       MQL = max(Q_1, Q_2, ..., Q_T)
    
    3. Minimum Queue Length (MinQL):
       MinQL = min(Q_1, Q_2, ..., Q_T)

Where:
    Q_t = Number of queued vehicles at step t
    T   = Total time steps

Unit: number of vehicles

Interpretation:
    - Lower values indicate better performance
    - MQL indicates worst-case congestion
    - Critical for preventing gridlock
"""
    
    @staticmethod
    def cumulative_reward() -> str:
        """4.2.3: Cumulative Reward definition."""
        return """
Section 4.2.3: Cumulative Reward
================================

Definition:
The sum of all rewards received by the agent during an episode.

Formula:
    R_total = Σ(t=1 to T) r_t

Where:
    r_t = Reward at time step t
    T   = Episode length

Reward Function (queue-delay):
    r_t = -(queue_length_t + waiting_time_t)

Alternative Reward (throughput):
    r_t = vehicles_passed_t - queue_penalty

Unit: dimensionless (typically negative for penalty-based rewards)

Interpretation:
    - Higher (less negative) values indicate better performance
    - Directly optimized by the RL agent
    - Reflects overall control strategy effectiveness
"""
    
    @staticmethod
    def print_all_definitions():
        """Print all metric definitions."""
        print(MetricDefinitions.average_vehicle_waiting_time())
        print(MetricDefinitions.queue_length())
        print(MetricDefinitions.cumulative_reward())

# Print metric definitions
print("="*60)
print("EVALUATION METRICS DEFINITIONS")
print("="*60)
MetricDefinitions.print_all_definitions()

EVALUATION METRICS DEFINITIONS

Section 4.2.1: Average Vehicle Waiting Time (AWT)

Definition:
The average time vehicles spend waiting at the intersection during an episode.

Formula:
    AWT = (1/T) × Σ(t=1 to T) W_t

Where:
    T  = Total number of time steps in episode
    W_t = Total waiting time of all vehicles at step t

Unit: seconds

Interpretation:
    - Lower values indicate better performance
    - Measures traffic flow efficiency
    - Directly impacts user experience


Section 4.2.2: Queue Length

Definition:
The number of vehicles stopped or moving very slowly (<0.1 m/s) at the intersection.

Variants:
    1. Average Queue Length (AQL):
       AQL = (1/T) × Σ(t=1 to T) Q_t

    2. Maximum Queue Length (MQL):
       MQL = max(Q_1, Q_2, ..., Q_T)

    3. Minimum Queue Length (MinQL):
       MinQL = min(Q_1, Q_2, ..., Q_T)

Where:
    Q_t = Number of queued vehicles at step t
    T   = Total time steps

Unit: number of vehicles

Interpretation:
    - Lower values indicate be

## Evaluation Data Classes

In [6]:
@dataclass
class EvaluationMetrics:
    """Container for evaluation metrics from a single episode."""
    episode_id: int
    avg_queue_length: float
    max_queue_length: float
    min_queue_length: float
    avg_waiting_time: float
    max_waiting_time: float
    avg_speed: float
    cumulative_reward: float
    total_vehicles_passed: int = 0
    queue_history: List[float] = field(default_factory=list)
    waiting_time_history: List[float] = field(default_factory=list)
    reward_history: List[float] = field(default_factory=list)
    
    def __str__(self) -> str:
        return f"""
Episode {self.episode_id} Results:
  Avg Queue Length:    {self.avg_queue_length:.2f} vehicles
  Max Queue Length:    {self.max_queue_length:.2f} vehicles
  Avg Waiting Time:    {self.avg_waiting_time:.2f} seconds
  Avg Speed:           {self.avg_speed:.2f} m/s
  Cumulative Reward:   {self.cumulative_reward:.2f}
  Vehicles Passed:     {self.total_vehicles_passed}
"""

print("✅ EvaluationMetrics dataclass defined")

✅ EvaluationMetrics dataclass defined


## Evaluation Runner

Main class to run evaluations of both DQN and Fixed-Time controllers.

In [7]:
class EvaluationRunner:
    """Run comprehensive evaluation of DQN vs Fixed-Time control strategies."""
    
    def __init__(
        self,
        scenario_path: str = "data/scenarios/hn_sample/config.sumocfg",
        num_episodes: int = 10,
        max_steps: int = 1000,
        warmup_steps: int = 60,
        output_dir: str = "outputs/chapter4"
    ):
        self.scenario_path = scenario_path
        self.num_episodes = num_episodes
        self.max_steps = max_steps
        self.warmup_steps = warmup_steps
        self.output_dir = output_dir
        
        Path(output_dir).mkdir(parents=True, exist_ok=True)

    def _make_env_cfg(self, config: dict) -> EnvConfig:
        """Build EnvConfig from loaded yaml config."""
        sumo_config = config.get('sumo', {})
        reward_config = config.get('reward', {})
        vn_config = config.get('vn_weights', {})
        reward_type = reward_config.get('type', 'queue_delay') if isinstance(reward_config, dict) else 'queue_delay'
        tls_id = sumo_config.get('tls_id', 'c') if isinstance(sumo_config, dict) else 'c'
        from src.env.sumo_env import VNWeights
        return EnvConfig(
            sumocfg_path=self.scenario_path,
            tls_id=tls_id,
            phases=[0, 1, 2, 3],
            step_length=1.0,
            action_duration=5,
            max_steps=self.max_steps,
            warmup_steps=self.warmup_steps,
            reward_type=reward_type,
            gui=False,
            vn_weights=VNWeights(
                motorcycle=vn_config.get('motorcycle', 0.5),
                car=vn_config.get('car', 1.5),
                bus=vn_config.get('bus', 2.0),
                truck=vn_config.get('truck', 2.0),
            ),
        )

    def evaluate_dqn(
        self,
        model_path: str,
        config_path: str = "config.yaml"
    ) -> List[EvaluationMetrics]:
        """Evaluate DQN agent over multiple episodes."""
        print("\n" + "="*60)
        print("EVALUATING DQN AGENT")
        print("="*60)
        
        with open(config_path, 'r') as f:
            config_data = yaml.safe_load(f)
            config = config_data if config_data is not None else {}
        
        env_cfg = self._make_env_cfg(config)

        # Detect actual state_dim from environment
        _tmp_env = SumoMDPEnv(env_cfg)
        _tmp_state = _tmp_env.reset()
        actual_state_dim = _tmp_state.shape[0]
        _tmp_env.close()
        print(f"  Detected state_dim = {actual_state_dim}")

        agent_cfg = AgentConfig(
            state_dim=actual_state_dim,
            action_dim=4,
        )
        agent = DQNAgent(agent_cfg)
        checkpoint = torch.load(model_path, map_location=agent.device)
        agent.q.load_state_dict(checkpoint)
        agent.q.eval()
        
        results = []
        
        for episode in range(self.num_episodes):
            print(f"\nRunning DQN Episode {episode + 1}/{self.num_episodes}...")
            
            env = SumoMDPEnv(env_cfg)
            state = env.reset()
            
            queue_history = []
            waiting_time_history = []
            reward_history = []
            speed_history = []
            total_reward = 0
            
            for step in range(self.max_steps):
                action = agent.act(state, eps=0.0)
                next_state, reward, done, info = env.step(action)
                
                if step >= self.warmup_steps:
                    queue_history.append(info.get("queue_length", 0.0))
                    waiting_time_history.append(info.get("avg_wait", 0.0))
                    speed_history.append(info.get("avg_speed", 0.0))
                    reward_history.append(reward)
                    total_reward += reward
                
                state = next_state
                if done:
                    break
            
            env.close()
            
            metrics = EvaluationMetrics(
                episode_id=episode + 1,
                avg_queue_length=float(np.mean(queue_history)) if queue_history else 0.0,
                max_queue_length=float(np.max(queue_history)) if queue_history else 0.0,
                min_queue_length=float(np.min(queue_history)) if queue_history else 0.0,
                avg_waiting_time=float(np.mean(waiting_time_history)) if waiting_time_history else 0.0,
                max_waiting_time=float(np.max(waiting_time_history)) if waiting_time_history else 0.0,
                avg_speed=float(np.mean(speed_history)) if speed_history else 0.0,
                cumulative_reward=float(total_reward),
                queue_history=queue_history,
                waiting_time_history=waiting_time_history,
                reward_history=reward_history
            )
            
            results.append(metrics)
            print(metrics)
        
        return results
    
    def evaluate_fixed_time(self, config_path: str = "config.yaml") -> List[EvaluationMetrics]:
        """Evaluate Fixed-Time controller over multiple episodes."""
        print("\n" + "="*60)
        print("EVALUATING FIXED-TIME CONTROLLER")
        print("="*60)
        
        with open(config_path, 'r') as f:
            config_data = yaml.safe_load(f)
            config = config_data if config_data is not None else {}
        
        env_cfg = self._make_env_cfg(config)
        results = []
        
        for episode in range(self.num_episodes):
            print(f"\nRunning Fixed-Time Episode {episode + 1}/{self.num_episodes}...")
            
            env = SumoMDPEnv(env_cfg)
            state = env.reset()
            
            controller = FixedTimeController(FixedTimeConfig(
                phase_schedule=[(2, 100), (3, 5), (0, 50), (1, 5)],
                action_duration=5,
            ))
            
            queue_history = []
            waiting_time_history = []
            reward_history = []
            speed_history = []
            total_reward = 0
            
            for step in range(self.max_steps):
                action = controller.get_action()
                next_state, reward, done, info = env.step(action)
                
                if step >= self.warmup_steps:
                    queue_history.append(info.get("queue_length", 0.0))
                    waiting_time_history.append(info.get("avg_wait", 0.0))
                    speed_history.append(info.get("avg_speed", 0.0))
                    reward_history.append(reward)
                    total_reward += reward
                
                state = next_state
                if done:
                    break
            
            env.close()
            
            metrics = EvaluationMetrics(
                episode_id=episode + 1,
                avg_queue_length=float(np.mean(queue_history)) if queue_history else 0.0,
                max_queue_length=float(np.max(queue_history)) if queue_history else 0.0,
                min_queue_length=float(np.min(queue_history)) if queue_history else 0.0,
                avg_waiting_time=float(np.mean(waiting_time_history)) if waiting_time_history else 0.0,
                max_waiting_time=float(np.max(waiting_time_history)) if waiting_time_history else 0.0,
                avg_speed=float(np.mean(speed_history)) if speed_history else 0.0,
                cumulative_reward=float(total_reward),
                queue_history=queue_history,
                waiting_time_history=waiting_time_history,
                reward_history=reward_history
            )
            
            results.append(metrics)
            print(metrics)
        
        return results
    
    def save_results(self, results: List[EvaluationMetrics], filename: str):
        """Save evaluation results to CSV."""
        output_path = Path(self.output_dir) / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                'Episode', 'Avg_Queue', 'Max_Queue', 'Min_Queue',
                'Avg_Wait_Time', 'Max_Wait_Time', 'Avg_Speed',
                'Cumulative_Reward', 'Vehicles_Passed'
            ])
            for m in results:
                writer.writerow([
                    m.episode_id, m.avg_queue_length, m.max_queue_length,
                    m.min_queue_length, m.avg_waiting_time, m.max_waiting_time,
                    m.avg_speed, m.cumulative_reward, m.total_vehicles_passed
                ])
        
        print(f"✅ Results saved to: {output_path}")


## Section 4.3: Statistical Analysis

Perform statistical tests to compare DQN vs Fixed-Time performance.

In [8]:
class StatisticalAnalysis:
    """Section 4.3.3: Statistical significance testing."""
    
    @staticmethod  # type: ignore[reportArgumentType]
    def compute_statistics(results: List[EvaluationMetrics]) -> Dict:
        """Compute mean and std for all metrics."""
        metrics = {
            'avg_queue': [r.avg_queue_length for r in results],
            'max_queue': [r.max_queue_length for r in results],
            'avg_wait': [r.avg_waiting_time for r in results],
            'avg_speed': [r.avg_speed for r in results],
            'cumulative_reward': [r.cumulative_reward for r in results]
        }
        
        stats_dict = {}
        for name, values in metrics.items():
            stats_dict[name] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'min': np.min(values),
                'max': np.max(values)
            }
        
        return stats_dict
    
    @staticmethod  # type: ignore[reportArgumentType]
    def paired_t_test(
        dqn_results: List[EvaluationMetrics],
        fixed_results: List[EvaluationMetrics]
    ) -> Dict:
        """Perform paired t-test on all metrics."""
        if not SCIPY_AVAILABLE:
            print("⚠️  scipy not available, skipping t-test")
            return {}
        
        results = {}
        
        metrics = {
            'avg_queue': ('avg_queue_length', 'lower'),
            'max_queue': ('max_queue_length', 'lower'),
            'avg_wait': ('avg_waiting_time', 'lower'),
            'avg_speed': ('avg_speed', 'higher'),
            'cumulative_reward': ('cumulative_reward', 'higher')
        }
        
        for metric_name, (attr_name, better) in metrics.items():
            dqn_values = [getattr(r, attr_name) for r in dqn_results]
            fixed_values = [getattr(r, attr_name) for r in fixed_results]
            
            # Perform t-test
            test_result = stats.ttest_rel(dqn_values, fixed_values)
            t_stat = float(test_result[0])  # Extract statistic from tuple
            p_value = float(test_result[1])  # Extract p-value from tuple
            
            results[metric_name] = {
                't_statistic': t_stat,
                'p_value': p_value,
                'significant': bool(p_value < 0.05),
                'dqn_better': bool((t_stat < 0 and better == 'lower') or (t_stat > 0 and better == 'higher'))
            }
        
        return results
    
    @staticmethod  # type: ignore[reportArgumentType]
    def cohen_d(group1: List[float], group2: List[float]) -> float:
        """Calculate Cohen's d effect size."""
        mean1, mean2 = np.mean(group1), np.mean(group2)
        std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
        n1, n2 = len(group1), len(group2)
        
        pooled_std = np.sqrt(((n1-1)*std1**2 + (n2-1)*std2**2) / (n1 + n2 - 2))
        return float((mean1 - mean2) / pooled_std) if pooled_std > 0 else 0.0
    
    @staticmethod  # type: ignore[reportArgumentType]
    def confidence_interval(data: List[float], confidence: float = 0.95) -> Tuple[float, float]:
        """Calculate confidence interval."""
        if not SCIPY_AVAILABLE:
            return (0.0, 0.0)
        
        mean = np.mean(data)
        sem = stats.sem(data)
        interval = sem * stats.t.ppf((1 + confidence) / 2, len(data) - 1)
        return (float(mean - interval), float(mean + interval))
    
    @staticmethod  # type: ignore[reportArgumentType]
    def print_comparison(
        dqn_results: List[EvaluationMetrics],
        fixed_results: List[EvaluationMetrics]
    ):
        """Print comprehensive statistical comparison."""
        print("\n" + "="*80)
        print("STATISTICAL COMPARISON: DQN vs FIXED-TIME")
        print("="*80)
        
        dqn_stats = StatisticalAnalysis.compute_statistics(dqn_results)
        fixed_stats = StatisticalAnalysis.compute_statistics(fixed_results)
        
        print("\nTable 4.6: Performance Comparison")
        print("-" * 80)
        print(f"{'Metric':<25} {'DQN':<20} {'Fixed-Time':<20} {'Improvement'}")
        print("-" * 80)
        
        metrics_display = [
            ('avg_queue', 'Avg Queue Length', 'vehicles'),
            ('max_queue', 'Max Queue Length', 'vehicles'),
            ('avg_wait', 'Avg Waiting Time', 'seconds'),
            ('avg_speed', 'Avg Speed', 'm/s'),
            ('cumulative_reward', 'Cumulative Reward', '')
        ]
        
        for key, label, unit in metrics_display:
            dqn_mean = dqn_stats[key]['mean']
            dqn_std = dqn_stats[key]['std']
            fixed_mean = fixed_stats[key]['mean']
            fixed_std = fixed_stats[key]['std']
            
            if key == 'avg_speed':
                improvement = ((dqn_mean - fixed_mean) / fixed_mean) * 100
            else:
                improvement = ((fixed_mean - dqn_mean) / fixed_mean) * 100
            
            dqn_str = f"{dqn_mean:.2f}±{dqn_std:.2f}"
            fixed_str = f"{fixed_mean:.2f}±{fixed_std:.2f}"
            impr_str = f"{improvement:+.1f}%"
            
            print(f"{label:<25} {dqn_str:<20} {fixed_str:<20} {impr_str}")

## Run Complete Evaluation

Execute the full evaluation pipeline.

In [9]:
# Close any existing SUMO connections
try:
    traci.close()
except:
    pass

# Configuration for demo (reduced episodes for quick testing)
NUM_EPISODES = 2  # Reduced for demo
MODEL_PATH = "outputs/dqn_vn_tls.pt"
SCENARIO_PATH = "data/scenarios/hn_sample/config.sumocfg"
OUTPUT_DIR = "outputs/chapter4"

# Create evaluation runner
runner = EvaluationRunner(
    scenario_path=SCENARIO_PATH,
    num_episodes=NUM_EPISODES,
    output_dir=OUTPUT_DIR
)

# Evaluate both strategies
print("\n🚀 Starting Chapter 4 Evaluation...\n")

# Check if model exists
if os.path.exists(MODEL_PATH):
    print(f"✅ Model found at: {MODEL_PATH}")
    dqn_results = runner.evaluate_dqn(MODEL_PATH)
    runner.save_results(dqn_results, "dqn_results.csv")
else:
    print(f"⚠️  Model not found at {MODEL_PATH}")
    print("Please train the model first using: ./run.sh train")
    dqn_results = None

fixed_results = runner.evaluate_fixed_time()
runner.save_results(fixed_results, "fixed_results.csv")

# Statistical comparison
if dqn_results:
    StatisticalAnalysis.print_comparison(dqn_results, fixed_results)

print("\n✅ Chapter 4 Evaluation Complete!")
print(f"Results saved to: {OUTPUT_DIR}")


🚀 Starting Chapter 4 Evaluation...

✅ Model found at: outputs/dqn_vn_tls.pt

EVALUATING DQN AGENT
  Detected state_dim = 32

Running DQN Episode 1/2...

Episode 1 Results:
  Avg Queue Length:    37.81 vehicles
  Max Queue Length:    93.00 vehicles
  Avg Waiting Time:    399.84 seconds
  Avg Speed:           4.62 m/s
  Cumulative Reward:   -12788.73
  Vehicles Passed:     0


Running DQN Episode 2/2...

Episode 2 Results:
  Avg Queue Length:    37.81 vehicles
  Max Queue Length:    93.00 vehicles
  Avg Waiting Time:    399.84 seconds
  Avg Speed:           4.62 m/s
  Cumulative Reward:   -12788.73
  Vehicles Passed:     0

✅ Results saved to: outputs\chapter4\dqn_results.csv

EVALUATING FIXED-TIME CONTROLLER

Running Fixed-Time Episode 1/2...

Episode 1 Results:
  Avg Queue Length:    46.26 vehicles
  Max Queue Length:    89.00 vehicles
  Avg Waiting Time:    233.73 seconds
  Avg Speed:           3.92 m/s
  Cumulative Reward:   -10865.02
  Vehicles Passed:     0


Running Fixed-Time Ep